# 05 Taxonomy of closed-reference OTUs

- **Input:** 
    - closed-reference OTU table 
    - sequences from `04_otu_clustering.ipynb`
    - reference `data/raw/ITS_database.fasta` 
<br><br>

- **Output:** (`processed_ignored/05_otu_taxonomy/`) 
    - OTU taxonomy
    - taxonomy-filtered OTU table and sequences
    
---
extended version

# 05 Taxonomy of Closed-Reference OTUs

**Creator:** Elettra

**Input:**<br>

`processed_ignored/04_otu/`
- 97% closed-reference OTU feature table (`otu_table_closed_ref_97.qza`)
- 97% closed-reference OTU representative sequences (`otu_seqs_closed_ref_97.qza`)
<br>

`raw/`
- UNITE ITS reference FASTA with taxonomy in the sequence headers (`ITS_database.fasta`)
- sample metadata for the taxonomic bar plots (`fungut_metadata.tsv`)

**Steps**

<table>
  <tr>
    <td>Extract UNITE taxonomy →</td>
    <td>Match OTU IDs and check for missing matches →</td>
    <td>Import taxonomy into QIIME 2 →</td>
    <td>Filter OTU table and sequences →</td>
    <td>Compare bar plots and check read retention</td>
  </tr>
</table>

**Output:**<br>

`processed_ignored/05_otu_taxonomy/`
- taxonomic assignments for closed-reference OTUs (`otu_taxonomy_closed_ref_97.qza`)
- taxonomy-filtered OTU feature table (`otu_table_closed_ref_97_filtered.qza`)
- taxonomy-filtered OTU representative sequences (`otu_seqs_closed_ref_97_filtered.qza`)
<br>

`processed/05_otu_taxonomy/`
- OTU taxonomy visualization (`otu_taxonomy_closed_ref_97.qzv`)
- taxonomic bar plot before filtering (`taxa_barplot_closed_ref_97.qzv`)
- taxonomic bar plot after filtering (`taxa_barplot_closed_ref_97_filtered.qzv`)

**Intermediate files:** (`processed_ignored/05_otu_taxonomy/`)
- matched OTU taxonomy in TSV format for QIIME 2 import (`otu_taxonomy_closed_ref_97.tsv`)

In [1]:
import os
import pandas as pd
from qiime2 import Visualization
from qiime2 import Artifact
import matplotlib.pyplot as plt 

%matplotlib inline

In [2]:
raw_dir = '../data/raw'
in_dir  = '../data/processed_ignored/04_otu'
out_dir = '../data/processed_ignored/05_otu_taxonomy'
vis_dir = '../data/processed/05_otu_taxonomy'

In [3]:
!mkdir -p $out_dir $vis_dir

Define input file locations: 

In [4]:
reference_fasta = f"{raw_dir}/ITS_database.fasta"
metadata        = f"{raw_dir}/fungut_metadata.tsv"
clustered_table = f"{in_dir}/otu_table_closed_ref_97.qza"
clustered_seqs  = f"{in_dir}/otu_seqs_closed_ref_97.qza"

Define the output file location:

In [5]:
taxonomy_tsv           = f"{out_dir}/otu_taxonomy_closed_ref_97.tsv"
taxonomy_output        = f"{out_dir}/otu_taxonomy_closed_ref_97.qza"
taxonomy_visualization = f"{vis_dir}/otu_taxonomy_closed_ref_97.qzv"
filtered_table         = f"{out_dir}/otu_table_closed_ref_97_filtered.qza"
filtered_seqs          = f"{out_dir}/otu_seqs_closed_ref_97_filtered.qza"
barplot                = f"{vis_dir}/taxa_barplot_closed_ref_97.qzv"
barplot_filtered       = f"{vis_dir}/taxa_barplot_closed_ref_97_filtered.qzv"

Inspect the FASTA headers to check for taxonomy labels and how they are separated from reference IDs:

In [6]:
! head -n 4 "$reference_fasta"

>UDB016649|k__Fungi;p__Basidiomycota;c__Agaricomycetes;o__Thelephorales;f__Thelephoraceae;g__Thelephora;s__Thelephora_albomarginata|SH1281904.10FU
TAGAGGAAGTAAAAGTCGTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTACCGAATTGTCAAACGGGTTGTTGCTGGTCTTCAAACGGGGACATGTGCACGCTCTGTTTACACATCCACTCACACCTGTGCACCCTCCGTAGTTCTATGGTCTGGGGGACCCTGTCTTCCTGCCGTGGTTCTACGTCTTTACACACACTCTGTAATAAAGTCTTATGGAATGTATGCCGCGTCTAACGCAATACAATACAACTTTCAGCAACGGATCTCTTGGCTCTCGCATCGATGAAGAACGCAGCGAAATGCGATAAGTAATGTGAATTGCAGAATTCAGTGAATCATCGAATCTTTGAACGCACCTTGCGCCCCTTGGCTATTCCGAGGGGCATGCCTGTTTGAGTATCATGAACACCTCAACTCTCATGGTTCGCCATGATGAGCTTGGACTCTGGGGGTTTTGCTGGCCTGTGGTCGGCTCCCCTCAAATGAATCGGCCTCCCAGTGTTTGGTGGCATCACGGGTGTGATAAATATCTACGCTCACGGTTGTCTGCCAGGTAACCTTTGGTGACAAAGGTTCGCTGGAGCTCACAGATGTCTCTCCTCGGCGAGGACAGCTTTTTTGAACGTTCGATCTCAAATCAGGTAGGACTACCCGCTGAACTTAAGCATATCAATA
>UDB016650|k__Fungi;p__Basidiomycota;c__Agaricomycetes;o__Thelephorales;f__Thelephoraceae;g__Tomentella;s__Tomentella_sp|SH1282297.10FU
TAGAGGAAGTAAAAGTCGTAACAAGGTTTCC

Read the reference FASTA into a pandas DataFrame using | as the column separator. Header lines of the FASTA file are divided into reference accession, taxonomy, and UNITE identifier.

In [7]:
reference = pd.read_csv(reference_fasta, sep="|", header=None)

In [8]:
reference.head()

,0,1,2
0,>UDB016649,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1281904.10FU
1,TAGAGGAAGTAAAAGTCGTAACAAGGTTTCCGTAGGTGAACCTGCG...,NaN,NaN
2,>UDB016650,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1282297.10FU
3,TAGAGGAAGTAAAAGTCGTAACAAGGTTTCCGTAGGTGAACCTGCG...,NaN,NaN
4,>UDB016651,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1146664.10FU


Keep only the header rows:

In [9]:
condition = reference[0].str.startswith(">")
headers = reference[condition].copy()
headers.head()

,0,1,2
0,>UDB016649,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1281904.10FU
2,>UDB016650,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1282297.10FU
4,>UDB016651,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1146664.10FU
6,>UDB016653,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1283391.10FU
8,>UDB016654,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...,SH1059744.10FU


Extract the taxonomic labels from column 1 into a pandas Series:

In [10]:
taxonomy = headers[1].copy()

Create feature IDs by joining the reference accession and kingdom label with |, without the leading >, so they match the IDs in our clustered table:

In [11]:
feature_ids = headers[0].str.removeprefix(">") + "|" + headers[1].str.split(";").str[0]

Create a table linking each feature ID to its taxonomic label, keeping the labels in a separate column so QIIME 2 can use them:

In [12]:
taxonomy_table = pd.DataFrame({"Feature ID": feature_ids, "Taxon": taxonomy})
taxonomy_table.head()

,Feature ID,Taxon
0,UDB016649|k__Fungi,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...
2,UDB016650|k__Fungi,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...
4,UDB016651|k__Fungi,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...
6,UDB016653|k__Fungi,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...
8,UDB016654|k__Fungi,k__Fungi;p__Basidiomycota;c__Agaricomycetes;o_...


Convert the count table from closed-reference clustering into a pandas DataFrame so we can match its feature IDs to the taxonomic labels in our reference database:

In [13]:
table_artifact = Artifact.load(clustered_table)
count_table = table_artifact.view(pd.DataFrame)
count_table.iloc[:3, :3] # visualize first three rows & columns 

,OQ099781|k__Fungi,KP035008|k__Fungi,KC143356|k__Fungi
ERR5327198,117.0,14.0,10.0
ERR5327199,9.0,7.0,4.0
ERR5327266,34489.0,7.0,8.0


Collect the feature IDs from the clustered count table so we can afterwards filter the reference taxonomy entries relevant to our samples:

In [14]:
clustered_ids = count_table.columns

Select rows from the reference taxonomy table whose feature IDs match those in our clustered feature table.

-> Reference IDs: taxonomy_table["Feature ID"]  
-> Clustered feature IDs: clustered_ids

In [15]:
condition = taxonomy_table["Feature ID"].isin(clustered_ids)
filtered_taxonomy_table = taxonomy_table[condition].copy()
filtered_taxonomy_table.iloc[:5, ]

,Feature ID,Taxon
9338,GU126458|k__Fungi,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__S...
9340,GU126457|k__Fungi,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__S...
9542,GU120193|k__Fungi,k__Fungi;p__Ascomycota;c__Eurotiomycetes;o__Eu...
11412,GQ849400|k__Fungi,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__S...
11592,GQ487292|k__Fungi,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__S...


Check that every clustered feature ID has a matching entry in the filtered taxonomy table (from the reference database):

In [16]:
missing_ids = clustered_ids.difference(filtered_taxonomy_table["Feature ID"])
print(f"Features without taxonomy: {len(missing_ids)}")

Features without taxonomy: 0


Save the filtered taxonomy table as a TSV file for import into QIIME 2:

In [17]:
filtered_taxonomy_table.to_csv(
    taxonomy_tsv, sep="\t", index=False, header=False
)

Import the saved taxonomy table as a QIIME 2 artifact:

In [18]:
! qiime tools import \
    --type 'FeatureData[Taxonomy]' \
    --input-format HeaderlessTSVTaxonomyFormat \
    --input-path "$taxonomy_tsv" \
    --output-path "$taxonomy_output"

Imported ../data/processed_ignored/05_otu_taxonomy/otu_taxonomy_closed_ref_97.tsv as HeaderlessTSVTaxonomyFormat to ../data/processed_ignored/05_otu_taxonomy/otu_taxonomy_closed_ref_97.qza


Visualize the taxonomy table to analyze which fungal names are linked to our clustered features:

In [19]:
! qiime metadata tabulate \
    --m-input-file "$taxonomy_output" \
    --o-visualization "$taxonomy_visualization"

Saved Visualization to: ../data/processed/05_otu_taxonomy/otu_taxonomy_closed_ref_97.qzv


In [20]:
! qiime taxa barplot \
    --i-table "$clustered_table" \
    --i-taxonomy "$taxonomy_output" \
    --m-metadata-file "$metadata" \
    --o-visualization "$barplot"

Saved Visualization to: ../data/processed/05_otu_taxonomy/taxa_barplot_closed_ref_97.qzv


Filter with the same criteria as the ASVs in `03_taxonomy.ipynb`, so both are comparable: keep features classified at least to class level (`c__`), remove `Unassigned` and `unidentified`. (Mitochondria filtering is only relevant for 16S data.)

In [21]:
! qiime taxa filter-table \
    --i-table "$clustered_table" \
    --i-taxonomy "$taxonomy_output" \
    --p-include c__ \
    --p-exclude Unassigned,unidentified \
    --o-filtered-table "$filtered_table"

! qiime taxa filter-seqs \
    --i-sequences "$clustered_seqs" \
    --i-taxonomy "$taxonomy_output" \
    --p-include c__ \
    --p-exclude Unassigned,unidentified \
    --o-filtered-sequences "$filtered_seqs"

Saved FeatureTable[Frequency] to: ../data/processed_ignored/05_otu_taxonomy/otu_table_closed_ref_97_filtered.qza
Saved FeatureData[Sequence] to: ../data/processed_ignored/05_otu_taxonomy/otu_seqs_closed_ref_97_filtered.qza


In [22]:
! qiime taxa barplot \
    --i-table "$filtered_table" \
    --i-taxonomy "$taxonomy_output" \
    --m-metadata-file "$metadata" \
    --o-visualization "$barplot_filtered"

Saved Visualization to: ../data/processed/05_otu_taxonomy/taxa_barplot_closed_ref_97_filtered.qzv


In [23]:
# reads kept per sample after taxonomy filtering
before = Artifact.load(clustered_table).view(pd.DataFrame).sum(axis=1)
after  = Artifact.load(filtered_table).view(pd.DataFrame).sum(axis=1)
retained = (after.reindex(before.index, fill_value=0) / before * 100)
retained.describe().round(1)

count    150.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
dtype: float64

In [ ]:
Visualization.load(barplot_filtered)